# 02 — eModel Optimisation

Build an `EModelOptimizationScanConfig`, expand it via `GridScanGenerationTask`, and
run the `EModelOptimizationTask` for each coordinate. The task seeds its working
directory from the previous stage's output, merges the optimisation-related
`pipeline_settings` from the blocks into `recipes.json`, then runs `pipeline.optimise(seed=...)`.

**Reads from:** `obi-output/01_efeature_extraction/grid_scan/0/`.

**Writes to:** `obi-output/02_emodel_optimization/grid_scan/0/` — the working directory
plus the new `checkpoints/` directory and the `run/<iteration_tag>/` archive.

The example uses **`optimiser='SO-CMA'`** with very small `max_ngen=2` and
`offspring_size=4`. Increase those for production runs.


## Imports

In [1]:
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.emodel_optimization._02_emodel_optimization.blocks import (
    OptimizationInitialize,
    OptimizationParams,
    OptimizationSettings,
)


## Build the scan config

In [2]:
previous_stage = (
    Path.cwd() / "../../../obi-output/01_efeature_extraction/grid_scan/0"
).resolve()
assert previous_stage.exists(), (
    f"{previous_stage} not found — run 01_efeature_extraction.ipynb first."
)

scan_config = obi.EModelOptimizationScanConfig(
    initialize=OptimizationInitialize(
        previous_stage_output_path=previous_stage,
        emodel="L5PC",
        species="rat",
        brain_region="SSCX",
    ),
    optimization_settings=OptimizationSettings(
        optimiser="SO-CMA",
        max_ngen=2,            # very small for demo runs
        optimisation_timeout=300.0,
        validation_threshold=5.0,
        seed=1,
    ),
    optimization_params=OptimizationParams(
        offspring_size=4,      # very small for demo runs
    ),
)
print(scan_config.optimization_settings.to_dict(scan_config.optimization_params))


{'optimiser': 'SO-CMA', 'max_ngen': 2, 'optimisation_timeout': 300.0, 'validation_threshold': 5.0, 'optimisation_params': {'offspring_size': 4}}


## Run the grid scan

In [3]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/02_emodel_optimization/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)


[2026-05-05 14:30:29,697] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/01_efeature_extraction/grid_scan/0/config -> /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0/config
[2026-05-05 14:30:29,698] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/01_efeature_extraction/grid_scan/0/morphologies -> /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0/morphologies
[2026-05-05 14:30:29,702] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/01_efeature_extraction/grid_scan/0/mechanisms -> /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0/mechanisms
[2026-05-05 14:30:29,787] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/01_efeature_extraction/grid_scan/0/ephys_data -> /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0/ephys_data
[2026-05-05 14:30:29,815] INFO: Seeded /Users/james/

--No graphics will be displayed.


[PosixPath('/Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0')]

## Inspect the checkpoints

In [4]:
coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)

checkpoints = sorted((coord_root / "checkpoints").rglob("*.pkl"))
print(f"Checkpoint files ({len(checkpoints)}):")
for p in checkpoints:
    print(" ", p.relative_to(coord_root))


Coordinate output: /Users/james/Documents/obi/code/obi-main/obi-output/02_emodel_optimization/grid_scan/0
Checkpoint files (1):
  checkpoints/L5PC/None/emodel=L5PC__species=rat__brain_region=SSCX__seed=1.pkl
